# MSDL-JCI — Thesis Defense Notebook

**Title:** Development of a Multi-Source Deep Learning Model for Stock Market Prediction on JCI
**Author:** Jordan Ferdinand (NIM: 23.K1.0018)
**Status:** Closed — rigorous negative result

This notebook visualizes the training dynamics, classification results, and trading simulation outcomes for the thesis defense.

## Setup

In [ ]:
from pathlib import Path
import sys

# Find project root: walk up from cwd until we find pyproject.toml
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 150,
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

sns.set_style("whitegrid")

def save_fig(fig, name):
    out = ROOT / "reports" / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches="tight", dpi=150)
    print(f"Saved: {out}")
    plt.close(fig)


# Architecture diagram
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis("off")

mods = ["Technical\n(LSTM, 64)", "Macro\n(MLP, 16)", "News\n(BERT→64)"]
mod_colors = ["#4C72B0", "#55A868", "#C44E52"]

for i, (m, c) in enumerate(zip(mods, mod_colors)):
    x = (i - 1) * 3
    ax.add_patch(plt.Rectangle((x, 3), 2.5, 1.2, facecolor=c, edgecolor="black", linewidth=1.5))
    ax.text(x + 1.25, 3.6, m, ha="center", va="center", fontsize=9, fontweight="bold", color="white")

ax.plot([-1, 1], [4.2, 4.2], "k-", linewidth=2)
ax.plot([1, 3], [4.2, 4.2], "k-", linewidth=2)
ax.plot([3, 5], [4.2, 4.2], "k-", linewidth=2)

ax.add_patch(plt.Rectangle((3.5, 2.5), 3, 1.2, facecolor="#DD8888", edgecolor="black", linewidth=1.5))
ax.text(5, 3.1, "Soft Gating\n(α, β, γ)", ha="center", va="center", fontsize=9, fontweight="bold")

ax.plot([5, 5], [3.7, 2.0], "k-", linewidth=2)
ax.add_patch(plt.Rectangle((4, 0.5), 2, 1.2, facecolor="#8888DD", edgecolor="black", linewidth=1.5))
ax.text(5, 1.1, "Fusion\n→ BCE", ha="center", va="center", fontsize=9)

ax.set_xlim(-2, 7)
ax.set_ylim(0, 6)
ax.set_title("Adaptive Soft Gating Fusion Architecture", fontsize=14, fontweight="bold", pad=20)
save_fig(fig, "architecture_overview.png")


In [ ]:
# Architecture diagram (matplotlib-based)
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis("off")

mods = ["Technical\n(LSTM, 64)", "Macro\n(MLP, 16)", "News\n(BERT→64)"]
mod_colors = ["#4C72B0", "#55A868", "#C44E52"]

for i, (m, c) in enumerate(zip(mods, mod_colors)):
    x = (i - 1) * 3
    ax.add_patch(plt.Rectangle((x, 3), 2.5, 1.2, facecolor=c, edgecolor="black", linewidth=1.5))
    ax.text(x + 1.25, 3.6, m, ha="center", va="center", fontsize=9, fontweight="bold", color="white")

# Concatenation
ax.plot([-1, 1], [4.2, 4.2], "k-", linewidth=2)
ax.plot([1, 3], [4.2, 4.2], "k-", linewidth=2)
ax.plot([3, 5], [4.2, 4.2], "k-", linewidth=2)

# Gating
ax.add_patch(plt.Rectangle((3.5, 2.5), 3, 1.2, facecolor="#DD8888", edgecolor="black", linewidth=1.5))
ax.text(5, 3.1, "Soft Gating\n(α, β, γ)", ha="center", va="center", fontsize=9, fontweight="bold")

# Fusion
ax.plot([5, 5], [3.7, 2.0], "k-", linewidth=2)
ax.add_patch(plt.Rectangle((4, 0.5), 2, 1.2, facecolor="#8888DD", edgecolor="black", linewidth=1.5))
ax.text(5, 1.1, "Fusion\n→ BCE", ha="center", va="center", fontsize=9)

ax.set_xlim(-2, 7)
ax.set_ylim(0, 6)
ax.set_title("Adaptive Soft Gating Fusion Architecture", fontsize=14, fontweight="bold", pad=20)
save_fig(fig, "architecture_overview.png")

# Training dynamics from ablation sweeps
train_log = pd.read_csv(ROOT / "reports" / "phase_06_training_dynamics" / "training_logs.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convert sweep values to numeric where possible
train_log["sweep_val"] = pd.to_numeric(train_log["sweep"], errors="coerce")

for config_val in train_log["config"].unique():
    subset = train_log[train_log["config"] == config_val]
    axes[0].plot(subset["sweep_val"], subset["val_auc"], "o-", label=f"config={config_val}")
    axes[1].plot(subset["sweep_val"], subset["val_mcc"], "o-", label=f"config={config_val}")

axes[0].set_xlabel("Hyperparameter Value")
axes[0].set_ylabel("Validation AUC")
axes[0].set_title("Val AUC vs Hyperparameter")
axes[0].set_xscale("log")
axes[0].legend(fontsize=8)
axes[1].set_xlabel("Hyperparameter Value")
axes[1].set_ylabel("Validation MCC")
axes[1].set_title("Val MCC vs Hyperparameter")
axes[1].set_xscale("log")
axes[1].legend(fontsize=8)

fig.suptitle("Training Dynamics: Hyperparameter Sweeps", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "training_dynamics.png")


In [ ]:
# Load training logs from ablation
train_log = pd.read_csv(ROOT / "reports" / "phase_06_training_dynamics" / "training_logs.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Val AUC by LR
for cfg in train_log["config"].unique():
    subset = train_log[train_log["config"] == cfg]
    axes[0].plot(subset["sweep"].astype(float), subset["val_auc"], "o-", label=f"{cfg}={subset.iloc[0]['config']}")

axes[0].set_xlabel("Learning Rate (log scale)")
axes[0].set_ylabel("Validation AUC")
axes[0].set_title("Val AUC vs Learning Rate")
axes[0].set_xscale("log")
axes[0].legend(fontsize=8)

# Val MCC by LR
for cfg in train_log["config"].unique():
    subset = train_log[train_log["config"] == cfg]
    axes[1].plot(subset["sweep"].astype(float), subset["val_mcc"], "o-", label=f"{cfg}")

axes[1].set_xlabel("Learning Rate (log scale)")
axes[1].set_ylabel("Validation MCC")
axes[1].set_title("Val MCC vs Learning Rate")
axes[1].set_xscale("log")
axes[1].legend(fontsize=8)

fig.suptitle("Training Dynamics: Hyperparameter Sweeps", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "training_dynamics.png")

# Classification results by model
cls = pd.read_csv(ROOT / "data" / "processed" / "ablation_classification_metrics.csv")

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cls))
width = 0.25

for i, col in enumerate(["Accuracy", "ROC-AUC", "F1-Score"]):
    vals = cls[col].values
    bars = ax.bar(x + i * width, vals, width, label=col)
    ax.bar_label(bars, fontsize=8, fmt="%.3f")

ax.set_xticks(x + width)
ax.set_xticklabels(cls["Model"], rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Classification Metrics by Model")
ax.legend()
ax.set_ylim(0, 1.0)
save_fig(fig, "classification_results.png")


In [ ]:
cls = pd.read_csv(ROOT / "data" / "processed" / "ablation_classification_metrics.csv")

# Radar-style comparison
metrics_cols = ["Accuracy", "ROC-AUC", "MCC"]
models = cls["Model"].tolist()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(models))
width = 0.25

for i, col in enumerate(["Accuracy", "ROC-AUC", "MCC"]):
    vals = cls[col].values
    bars = ax.bar(x + i * width, vals, width, label=col)
    ax.bar_label(bars, fontsize=8, fmt="%.3f")

ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Classification Metrics by Model")
ax.legend()
ax.set_ylim(0, 1.0)
save_fig(fig, "classification_results.png")

In [ ]:
# Confusion-matrix heatmap for the best model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load baseline classification data
baseline_cls = pd.read_csv(ROOT / "reports" / "phase_08_baselines" / "baseline_classification.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy ranking
top = baseline_cls[baseline_cls["Model"].isin(["always-positive", "always-negative"]) == False]
top = top.sort_values("Accuracy", ascending=False).head(8)
colors = sns.color_palette("viridis", len(top))
bars = axes[0].barh(range(len(top)), top["Accuracy"], color=colors)
axes[0].set_yticks(range(len(top)))
axes[0].set_yticklabels(top["Model"], fontsize=9)
axes[0].set_xlabel("Accuracy (%)\n" if False else "Accuracy")
axes[0].set_title("Top Model Accuracies")
for i, (v, p) in enumerate(zip(top["Accuracy"], top["Precision"])):
    axes[0].text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)

# MCC ranking
bars = axes[1].barh(range(len(top)), top["MCC"], color=colors)
axes[1].set_yticks(range(len(top)))
axes[1].set_yticklabels(top["Model"], fontsize=9)
axes[1].set_title("Top Model MCC Scores")
for i, v in enumerate(top["MCC"]):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

fig.suptitle("Baseline Classification Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "baseline_classification.png")

---

## 4. Gating Weight Analysis

In [ ]:
# Gating weight analysis
gw = pd.read_csv(ROOT / "data" / "processed" / "gating_weights_summary.csv")

fig, ax = plt.subplots(figsize=(8, 5))
colors_gw = ["#4C72B0", "#55A868", "#C44E52"]
bars = ax.bar(gw["Weight Parameter"], gw["Mean Weight"], color=colors_gw, edgecolor="black")
ax.bar_label(bars, fmt="%.3f", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Gating Weight")
ax.set_title("Average Soft Gating Weights\n(Technical α, Macro β, News γ)")
ax.set_ylim(0, 1)
save_fig(fig, "gating_weights.png")


---

## 5. Trading Simulation — Financial Results

In [ ]:
# Trading simulation results
fin = pd.read_csv(ROOT / "data" / "processed" / "ablation_financial_metrics.csv")
gn = pd.read_csv(ROOT / "reports" / "phase_10_trading" / "gross_vs_net_results.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = fin["Model"].tolist()
x = np.arange(len(models))
width = 0.35

# Total Return comparison
axes[0].bar(x - width/2, fin["Total Return (%)"], width, label="Total Return", color="#4C72B0")
axes[0].bar(x + width/2, fin["Benchmark Return (%)"], width, label="Buy-and-Hold", color="#AAAAAA")
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=30, ha="right", fontsize=8)
axes[0].set_ylabel("Return (%)")
axes[0].set_title("Strategy vs Buy-and-Hold")
axes[0].legend(fontsize=8)

# Sharpe Ratio
bars = axes[1].bar(x, fin["Sharpe Ratio"], color="#55A868", edgecolor="black")
axes[1].bar_label(bars, fmt="%.3f", fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=30, ha="right", fontsize=8)
axes[1].set_ylabel("Sharpe Ratio")
axes[1].set_title("Annualized Sharpe Ratio")

fig.suptitle("Trading Simulation: Financial Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "trading_financial.png")


In [ ]:
# Gross vs Net with cost drag
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(gross_net))
width = 0.35
bars1 = ax.bar(x - width/2, gross_net["gross"], width, label="Gross Return", color="#4C72B0")
bars2 = ax.bar(x + width/2, gross_net["net"], width, label="Net Return", color="#C44E52")
ax.bar_label(bars1, fmt="%.2f%%", fontsize=9)
ax.bar_label(bars2, fmt="%.2f%%", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(gross_net["Model"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Return (%)")
ax.set_title("Gross vs Net Returns (with transaction costs)")
ax.legend()

# Cost drag
ax2 = ax.twinx()
ax2.plot(x, gross_net["cost_drag_pp"], "D-", color="black", label="Cost Drag (pp)", markersize=6)
ax2.set_ylabel("Cost Drag (percentage points)", fontsize=10)
ax2.legend(loc="upper left")

plt.tight_layout()
save_fig(fig, "gross_vs_net.png")

# Gross vs Net with cost drag
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(gn))
width = 0.35
bars1 = ax.bar(x - width/2, gn["gross"], width, label="Gross Return", color="#4C72B0")
bars2 = ax.bar(x + width/2, gn["net"], width, label="Net Return", color="#C44E52")
ax.bar_label(bars1, fmt="%.2f%%", fontsize=9)
ax.bar_label(bars2, fmt="%.2f%%", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(gn["Model"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Return (%)")
ax.set_title("Gross vs Net Returns (with transaction costs)")
ax.legend()

ax2 = ax.twinx()
ax2.plot(x, gn["cost_drag_pp"], "D-", color="black", label="Cost Drag (pp)", markersize=6)
ax2.set_ylabel("Cost Drag (percentage points)", fontsize=10)
ax2.legend(loc="upper left")

plt.tight_layout()
save_fig(fig, "gross_vs_net.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Load cumulative returns plots if available as data
try:
    cum = pd.read_csv(ROOT / "reports" / "phase_10_trading" / "gross_vs_net_results.csv")
except Exception:
    cum = None

# Plot from processed data
if (ROOT / "data" / "processed" / "cumulative_returns_comparison.png").exists():
    from PIL import Image
    img = Image.open(ROOT / "data" / "processed" / "cumulative_returns_comparison.png")
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title("Cumulative Returns (from run)")

# Drawdown visualization
axes[1].bar(gross_net["Model"], gross_net["maxdd"], color="#C44E52", edgecolor="black")
axes[1].bar_label(axes[1].containers[0], fmt="%.2f%%", fontsize=9)
axes[1].set_xticklabels(gross_net["Model"], rotation=30, ha="right", fontsize=9)
axes[1].set_ylabel("Max Drawdown (%)")
axes[1].set_title("Maximum Drawdown by Strategy")

fig.suptitle("Equity Curve Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "equity_curves.png")

---

## 7. Key Findings Summary

In [ ]:
# Summary table
summary_data = {
    "Finding": [
        "Proposed Soft Gating collapsed (MCC ≈ 0)",
        "Branch AUC ≈ 0.50 (no signal)",
        "Gating weights balanced but not informative",
        "Best heuristic: Momentum-5d",
        "Conclusion: Negative result"
    ],
    "Evidence": [
        "MCC near zero, probability collapse",
        "Branch probes showed AUC ≈ 0.50",
        "Entropy improved but MCC unchanged",
        "Net ≈ 28.07, Sharpe ≈ 1.31",
        "Nothing deployable for live trading"
    ],
    "Status": [
        "✗ Not viable",
        "✗ No predictive edge",
        "✗ Weight ≠ Signal",
        "△ Needs validation",
        "📋 Closed — rigorous negative"
    ]
}
summary_df = pd.DataFrame(summary_data)
summary_df.style.set_properties(**{"text-align": "left"})
print(summary_df.to_string(index=False))

# Key findings summary
summary_data = {
    "Finding": [
        "Proposed Soft Gating collapsed",
        "Branch AUC ~0.50 (no signal)",
        "Gating balanced but not informative",
        "Best heuristic: Momentum-5d",
        "Conclusion: Negative result"
    ],
    "Evidence": [
        "MCC near zero",
        "Branch probes showed AUC ~0.50",
        "Entropy improved, MCC unchanged",
        "Net ~28.07, Sharpe ~1.31",
        "Nothing deployable for live trading"
    ],
    "Status": [
        "Not viable",
        "No predictive edge",
        "Weight != Signal",
        "Needs validation",
        "Closed — rigorous negative"
    ]
}
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
